# 12. Sensor Frequency Analysis & Driving Event Characterization

**Objective**: Compute single-sided FFT magnitude spectra, Welch Power Spectral Density (PSD), and characterize 7 representative driving regimes.

## 1. Setup & Environment
Load sequence S1 and spectral analysis module.

In [ ]:
import sys
sys.path.insert(0, '../..')
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Data_details.src.dataset_loader import DatasetLoader
from Data_details.src.spectral_analysis import compute_fft_spectrum, compute_welch_psd, segment_driving_events, build_frequency_characterization_table

with open('../config/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
loader = DatasetLoader('../../IO-VNBD-master', '../../Data_details/data/raw')
s_df, _ = loader.load_sequence(cfg['selected_sequence']['smartphone_file'])
print(f'Loaded Sequence S1: {len(s_df)} rows')


## 2. Event Segmentation & Frequency Table
Segment driving events (Stationary, Braking, Cornering, Cruising, Bumps) and extract spectral metrics.

In [ ]:
events = segment_driving_events(s_df)
freq_df = build_frequency_characterization_table(s_df, events, fs=10.0)
display(freq_df[['event_name', 'accel_dominant_freq_hz', 'accel_hf_energy_ratio_pct', 'peak_accel_mps2', 'crest_factor']])


## 3. FFT Magnitude & Welch PSD
Plot single-sided spectrum and inspect Nyquist boundary at 5.0 Hz.

In [ ]:
ay = s_df['acc_y'].values
freqs, mag, pwr = compute_fft_spectrum(ay, fs=10.0)
f_psd, psd = compute_welch_psd(ay, fs=10.0)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(freqs, mag, color='#2ecc71', lw=1.2)
axes[0].set_title('Forward Acceleration FFT Magnitude Spectrum')
axes[0].set_ylabel('Magnitude (m/s²)')
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(f_psd, psd, color='#e74c3c', lw=1.2)
axes[1].set_title('Welch Power Spectral Density (PSD)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('PSD ((m/s²)²/Hz)')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Key Findings
- Useful vehicle maneuvers concentrate below 1.5 Hz.
- Frequencies above 2.5 Hz are dominated by engine vibration and aliased road chatter.